### Carregando bibliotecas e dados

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
#pd.set_option('display.max_rows', None)
#pd.set_option('display.max_columns', None)

pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')

In [6]:
df = pd.read_csv("data.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'data.csv'

In [ ]:
df_soc = pd.read_csv("societies.csv")[["ID", "Name"]].iloc[:339].copy()
df_soc

In [ ]:
df_B009 = df[df["Var_ID"] == "B009"].copy().reset_index(drop=True)
df_B009.drop(columns=["ID", "Code_ID", "Source", "source_coded_data", "admin_comment"], inplace=True)

df_B021 = df[(df["Var_ID"] == "B021") & (df["Value"].notna())].copy().reset_index(drop=True)
df_B021.drop(columns=["ID", "Code_ID", "Source", "source_coded_data", "admin_comment"], inplace=True)

**Obs:** \
B009 se refere ao *nomadismo* ou *sedentarismo* da sociedade \
B021 se refere à *idade média* ao casar das mulheres

In [ ]:
df_B009

In [ ]:
df_B021

In [ ]:
print(f"Valores duplicados B009: {df_B009["Soc_ID"].duplicated().sum()}")
print(f"Valores duplicados B021: {df_B021["Soc_ID"].duplicated().sum()}")

In [ ]:
for _, row in df_B021.dropna(subset=["Comment"]).iterrows():
    print(f"Society {row['Soc_ID']}")
    print(f"Comment: {row['Comment']}\n")

#### Lidando com sociedades com valores duplicados

In [ ]:
df_B021["Value"] = pd.to_numeric(df_B021["Value"])
df_B021_corrigido = df_B021.groupby(["Soc_ID", "sub_case"], dropna=False, as_index=False)["Value"].mean()
df_B021_corrigido

In [ ]:
print(f"Valores duplicados B021: {df_B021_corrigido["Soc_ID"].duplicated().sum()}")

### Criando conjunto de dados para teste

In [ ]:
### Juntando as variáveis
df_teste = pd.merge(
    df_B009[["Soc_ID", "Value"]],
    df_B021_corrigido[["Soc_ID", "sub_case", "Value"]],
    on="Soc_ID",
    how="inner",
    suffixes=("_B009", "_B021"),
    validate="one_to_many"
)

In [ ]:
### Nomeando as sociedades e as variáveis
df_teste = df_teste.merge(
    df_soc,
    left_on="Soc_ID",
    right_on="ID",
    how="left",
)
df_teste = df_teste[["Name", "sub_case", "Value_B009", "Value_B021"]]
df_teste.columns = ["Society", "sub_case", "Mobility", "Avg age of marriage of women"]
df_teste

### Análise descritiva

#### Estatísticas descritivas das variáveis

In [ ]:
df_teste.head()

In [ ]:
# Missing values
print(df_teste.isnull().sum())

In [ ]:
print(df_teste["Avg age of marriage of women"].describe())

In [ ]:
print(df_teste["Mobility"].describe())

In [ ]:
print(df_teste.loc[df_teste["Mobility"] == "Sedentary", "Avg age of marriage of women"].describe())

In [ ]:
print(df_teste.loc[df_teste["Mobility"] == "Mobile", "Avg age of marriage of women"].describe())

#### Gráficos das variáveis

In [ ]:
### Histograma geral

plt.figure()
plt.hist(df_teste["Avg age of marriage of women"], density=True)
plt.xlabel("Idade média (anos) de mulheres no primeiro casamento")
plt.ylabel("Frequência")
plt.title("Distribuição geral")
plt.show()

In [ ]:
### Boxplot geral

plt.figure()
sns.boxplot(df_teste["Avg age of marriage of women"], width=0.4)
sns.stripplot(df_teste["Avg age of marriage of women"], alpha=0.5)
plt.title("Boxplot geral")
plt.ylabel('Idade média (anos) de mulheres no primeiro casamento')
plt.grid(linestyle='--', alpha=0.6)
plt.show()

In [ ]:
### histogramas por mobilidade da sociedade

plt.figure()

plt.hist(df_teste.loc[df_teste["Mobility"] == "Sedentary", "Avg age of marriage of women"],
         density=True,
         color="green",
         alpha=0.7,
         label="Sedentária"
         )
plt.hist(df_teste.loc[df_teste["Mobility"] == "Mobile", "Avg age of marriage of women"],
         density=True,
         color="red",
         alpha=0.7,
         label="Móvel"
         )

plt.xlabel("Idade média (anos) de mulheres no primeiro casamento")
plt.ylabel("Frequência")
plt.title("Distribuição por mobilidade da sociedade")
plt.legend()
plt.show()

In [ ]:
### Boxplot por mobilidade da sociedade

plt.figure()

sns.boxplot(
            data=df_teste,
            x="Mobility",
            y="Avg age of marriage of women",
            order=["Sedentary", "Mobile"],
            hue="Mobility",
            palette=["red", "green"],
            width=0.4,
            boxprops={"alpha": 0.7}
)

sns.stripplot(
            data=df_teste,
            x="Mobility",
            y="Avg age of marriage of women",
            order=["Sedentary", "Mobile"],
            hue="Mobility",
            palette=["orange"],
            alpha=0.7
)

plt.title("Boxplots por mobilidade da sociedade")
plt.xticks(
    ticks=[0, 1],
    labels=["Sedentária", "Móvel"]
)
plt.xlabel("Mobilidade da sociedade")
plt.ylabel("Idade média (anos) de mulheres no primeiro casamento")
plt.grid(linestyle='--', alpha=0.6)
plt.show()

# Testes de Hipótese e Intervalos de Confiança


#### Teste Bootstrap de Igualdade entre variâncias

In [ ]:
idades_nomades = df_teste.loc[
    df_teste["Mobility"] == "Mobile",
    "Avg age of marriage of women"
].dropna().to_numpy()

idades_sedentarias = df_teste.loc[
    df_teste["Mobility"] == "Sedentary",
    "Avg age of marriage of women"
].dropna().to_numpy()

import numpy as np

# Vetores com as idades ao casar de cada grupo
nomades = np.asarray(idades_nomades, dtype=float)
sedentarias = np.asarray(idades_sedentarias, dtype=float)

# Estatística observada: diferença entre as variâncias
T_obs = np.var(nomades, ddof=1) - np.var(sedentarias, ddof=1)

# Variância comum estimada sob H0
n1, n2 = len(nomades), len(sedentarias)

media1, media2 = nomades.mean(), sedentarias.mean()

res1 = nomades - media1
res2 = sedentarias - media2

variancia_comum = (np.sum(res1**2) + np.sum(res2**2)) / (n1 + n2 - 2)

# Ajusta os resíduos para impor a mesma variância populacional
res1 = res1 * np.sqrt(variancia_comum / np.var(res1, ddof=0))
res2 = res2 * np.sqrt(variancia_comum / np.var(res2, ddof=0))

# Bootstrap sob H0
B = 9999
rng = np.random.default_rng(42)

T_boot_var = np.empty(B)

for b in range(B):
    amostra1 = rng.choice(res1, size=n1, replace=True)
    amostra2 = rng.choice(res2, size=n2, replace=True)

    T_boot_var[b] = np.var(amostra1, ddof=1)- np.var(amostra2, ddof=1)

# Valor-p bilateral
p_valor = (
    1 + np.sum(np.abs(T_boot_var) >= abs(T_obs))
) / (B + 1)

print("Variância dos nômades:", np.var(nomades, ddof=1))
print("Variância dos sedentários:", np.var(sedentarias, ddof=1))
print("Diferença observada:", T_obs)
print("Valor-p bootstrap:", p_valor)

#### Testes de Permutação para Estatísticas Variadas

In [ ]:
# Estatísticas observadas originalmente
T_media = nomades.mean() - sedentarias.mean()

T_mediana = np.median(nomades) - np.median(sedentarias)

T_HL = np.median(nomades[:, None] - sedentarias[None, :])

# Teste de permutação
B = 99999
rng = np.random.default_rng(67)

dados = np.concatenate([nomades, sedentarias])
n_nomades = len(nomades)

T_perm_media = np.empty(B)
T_perm_mediana = np.empty(B)
T_perm_HL = np.empty(B)

for b in range(B):
    permutados = rng.permutation(dados)

    grupo_n = permutados[:n_nomades]
    grupo_s = permutados[n_nomades:]

    T_perm_media[b] = grupo_n.mean() - grupo_s.mean()
    T_perm_mediana[b] = (np.median(grupo_n) - np.median(grupo_s))
    T_perm_HL[b] = np.median(grupo_n[:, None] - grupo_s[None, :])

# Valores-p bilaterais
def valor_p_permutacao(T_obs, T_perm):
    return (
        1 + np.sum(np.abs(T_perm) >= abs(T_obs))
    ) / (len(T_perm) + 1)

p_media = valor_p_permutacao(T_media, T_perm_media)
p_mediana = valor_p_permutacao(T_mediana, T_perm_mediana)
p_HL = valor_p_permutacao(T_HL, T_perm_HL)

# Resultados
print("Diferença de médias:", T_media)
print("Valor-p (médias):", p_media)

print("\nDiferença de medianas:", T_mediana)
print("Valor-p (medianas):", p_mediana)

print("\nEstimador de Hodges-Lehmann:", T_HL)
print("Valor-p (Hodges-Lehmann):", p_HL)

print("\n \nMédia dos nômades:", nomades.mean())
print("Média das sedentárias:", sedentarias.mean())
print("Diferença observada:", T_obs)
print("Valor-p do teste de permutação:", p_valor)

#### IC 95% Para a Diferença Entre Médias

In [ ]:
diferencas_boot = np.empty(B)

for b in range(B):
    amostra_n = rng.choice(nomades, size=len(nomades), replace=True)
    amostra_s = rng.choice(sedentarias, size=len(sedentarias), replace=True)

    diferencas_boot[b] = amostra_n.mean() - amostra_s.mean()

IC_95 = np.percentile(diferencas_boot, [2.5, 97.5])

print("Diferença de médias:", nomades.mean() - sedentarias.mean())
print("IC de 95%:", IC_95)